# CryoTomoSim-replica specimen generator

A completely self-contained specimen generator -- `CryoTomoSimSpecimenGenerator` -- that replicates [CryoTomoSim (CTS)](https://doi.org/10.1101/2023.04.28.538636)'s own algorithms (particle placement, organic alpha-shape-based membrane shapes, carbon support film, gold fiducial beads), reimplemented from scratch in NumPy/PyTorch.

**No polnet, no VTK.** This is a separate, independent pipeline from `specimen/cryoet.py`'s polnet-based `CryoETSpecimenGenerator` (see that notebook, `cryoet-specimen-generator.ipynb`, for the polnet-backed alternative) -- the two share no code.

Two pieces of specter's existing, physically-accurate infrastructure ARE reused here (both plain physics utilities, neither polnet-related):
- `specter.potential.PotentialBuilder` -- every real protein species' density comes from actual PDB atomic coordinates via Kirkland/Lobato scattering potentials.
- `specter.ice.RandomIcemaker` -- the ice/solvent layer uses real Kirkland-oxygen-potential-convolved amorphous ice.

Membrane geometry and carbon/bead bulk density are new, self-contained code -- see `specter/specimen/_cts_membrane.py`, `_cts_placement.py`, `_cts_grid.py`, `cryotomosim.py` for the implementation and docstrings on exactly what's ported vs. simplified vs. CTS's own MATLAB source.

In [ ]:
import matplotlib.pyplot as plt

from specter.specimen.cryotomosim import (
    BeadSpec,
    CryoTomoSimSpecimenGenerator,
    GridSpec,
    MembraneSpec,
    ProteinSpec,
)

## Configure the specimen

- `protein_specs`: one `ProteinSpec` per protein species -- `pdb_source` is a 4-character PDB ID (fetched from RCSB and cached) or a local file path, `max_count` the number of copies to attempt to place.
- `membrane_specs`: one `MembraneSpec` per organic-blob vesicle population.
- `bead_spec` / `grid_spec`: optional gold fiducial beads / carbon support film.

This example keeps the volume small (fast to run as a demo); scale `target_shape` up for a production-sized specimen.

In [ ]:
protein_specs = [
    ProteinSpec(pdb_source="1mbo", max_count=6),
]

membrane_specs = [
    MembraneSpec(count=1, size=180.0, roughness=0.65, thickness=40.0),
]

gen = CryoTomoSimSpecimenGenerator(
    protein_specs=protein_specs,
    membrane_specs=membrane_specs,
    bead_spec=BeadSpec(radii=[60.0], count_per_radius=1),
    grid_spec=GridSpec(thickness=120.0, hole_radius=300.0),
    target_shape=(64, 128, 128),  # (Z, Y, X)
    v_size=8.0,  # Angstrom/voxel
    ice_opacity=1.0,
    density_cutoff=0.35,
    seed=0,
)

## Generate

Returns a `torch.Tensor`, specter-native axis order `(Z, Y, X)`, in scattering-potential units -- the same contract as `specimen.from_volume.load_specimen_volume` and `CryoETSpecimenGenerator.generate()`, so it plugs directly into `TiltSeriesGenerator`/`MicrographGenerator`/`ImageGenerator`.

In [ ]:
import time

t0 = time.time()
volume = gen.generate()
print(f"generated in {time.time() - t0:.1f}s")
print(
    "shape:",
    tuple(volume.shape),
    "nonzero voxels:",
    int((volume > 0).sum()),
    "max:",
    float(volume.max()),
)
print("placements:", len(gen.placements))
for p in gen.placements:
    print(" ", p.species_id, tuple(p.center_zyx.tolist()))

## Visualize a mid-slice

In [ ]:
mid_z = volume.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(volume[mid_z].numpy(), cmap="gray")
axes[0].set_title(f"XY slice at Z={mid_z}")
axes[1].imshow(volume[:, volume.shape[1] // 2, :].numpy(), cmap="gray")
axes[1].set_title("XZ slice at mid-Y")
axes[2].imshow(volume.sum(dim=0).numpy(), cmap="gray")
axes[2].set_title("Z-projection")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()